In [1]:
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

# 設定 Request Headers，模擬瀏覽器行為以防被阻擋
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
}

# 主網址與即時新聞首頁
base_url = "https://udn.com"
start_url = "https://udn.com/news/breaknews/1"

print("開始讀取即時新聞主頁並擷取主題網址\n")

# 1. 發送請求取得即時新聞主頁 HTML
response = requests.get(start_url, headers=headers)

if response.status_code != 200:
    print(f"無法連線至網站，錯誤代碼: {response.status_code}")
else:
    print('連線成功！！！\n')
    soup = BeautifulSoup(response.text, "html.parser")

    # 2. 定義我們要找的目標主題
    target_categories = ["娛樂", "社會", "運動", "科技", "文教"]
    category_links = {}

    # 找出所有的 <a> 標籤
    all_links = soup.find_all("a")

    # 3. 使用正規表達式 (Regular Expression) 匹配符合格式的網址，並篩選出目標主題
    # 預期格式如: /news/breaknews/1/8#breaknews
    pattern = re.compile(r"/news/breaknews/1/\d+#breaknews")

    for link in all_links:
        href = link.get("href", "")
        text = link.get_text(strip=True)

        if text in target_categories and pattern.match(href):
            # 組合出完整的 URL
            category_links[text] = base_url + href

    print("成功使用正規表達式擷取到目標主題網址：")
    for cat, url in category_links.items():
        print(f"【{cat}】: {url}")

    print("\n開始爬取各主題新聞標題並轉換為 DataFrame")

    # 4. 巡迴每個主題網址，爬取新聞標題
    for category, url in category_links.items():
        print(f"\n正在處理【{category}】主題...")

        cat_response = requests.get(url, headers=headers)
        if cat_response.status_code != 200:
            print(f"無法讀取 {category} 的內容")
            continue

        cat_soup = BeautifulSoup(cat_response.text, "html.parser")

        # 鎖定聯合新聞網的標題標籤
        title_tags = cat_soup.find_all(
            ["h2", "h3"], class_=["story-list__heading", "story-list__title"]
        )

        titles_list = []
        for tag in title_tags:
            a_tag = tag.find("a")
            if a_tag:
                title_text = a_tag.get_text(strip=True)
            else:
                title_text = tag.get_text(strip=True)

            if title_text:
                titles_list.append(title_text)

        # 備用機制：如果上面的 class 報社有微調，則抓取另一種常見的 CSS 選擇器結構
        if not titles_list:
            news_items = cat_soup.select(
                ".story-list__news .story-list__text h2 a"
            )
            titles_list = [item.get_text(strip=True) for item in news_items]

        # 5. 將結果整理成 Pandas DataFrame 並呈現
        df = pd.DataFrame(titles_list, columns=[f"{category}新聞標題"])

        print(f"【{category}】即時新聞結果 (共 {len(df)} 筆)：")
        if not df.empty:
            display(df)  # Jupyter 專用美化輸出表格
        else:
            print("目前該分頁沒有擷取到新聞標題。")

開始讀取即時新聞主頁並擷取主題網址

連線成功！！！

成功使用正規表達式擷取到目標主題網址：
【娛樂】: https://udn.com/news/breaknews/1/8#breaknews
【社會】: https://udn.com/news/breaknews/1/2#breaknews
【運動】: https://udn.com/news/breaknews/1/7#breaknews
【科技】: https://udn.com/news/breaknews/1/13#breaknews
【文教】: https://udn.com/news/breaknews/1/12#breaknews

開始爬取各主題新聞標題並轉換為 DataFrame

正在處理【娛樂】主題...
【娛樂】即時新聞結果 (共 20 筆)：


,娛樂新聞標題
0,哭了！粉絲擠破玻璃門見血！張凌赫活動喊卡霸氣賠機票住宿全包了
1,恭喜！玖壹壹洋蔥首曝婚紗照 絕美愛妻超仙正面引暴動
2,43歲王嫚萱報喜懷雙胞胎！挺過孕初期出血甜喊：雙倍幸福！
3,韋禮安突破唱跳迷倒萬人 模仿張學友出包了
4,張凌赫現身爆「喪屍級」暴動！粉苦等14小時擠爆玻璃門見血急喊卡
5,72歲陳盈潔生命垂危！許常德爆前夫心碎告別 後事內幕曝光
6,國民天團27年傳奇落幕！粉絲東京巨蛋淚別ARASHI嵐
7,嵐26年傳奇落幕！團體活動結束 回顧3次來台經典畫面
8,阿本「罹腦動脈瘤」3年！坦承接受性向、感情 最新病況曝：很幸運活著
9,擁5塊農地身價破億？王中平挺余皓然新書露面全說了



正在處理【社會】主題...
【社會】即時新聞結果 (共 20 筆)：


,社會新聞標題
0,全國首例空拍機投毒！宜蘭張美阿嬤兒子無差別危害動物 遭判刑1年8月
1,他酒後嬉鬧...基隆廟口當街連噴辣椒水 害路人店內咳嗽、流淚
2,4天登門7次！怪伯鎖定宜蘭落單闆娘「摸褲襠聊性事」警火速逮人
3,逃逸移工變車手 台中超商領款可疑被逮
4,「卡米地」事件疑似中共跨境鎮壓 港人憂越來越頻繁
5,影／台南人潮眾多海安商圈餐廳火警濃煙漫天 消防派遣25車60人急救援
6,男子基隆廟口噴辣椒水 警方逮人移請法院裁罰
7,影／竹東夜市旁驚見「推土機式駕駛」 白車硬擠機車陣一路撞
8,國父紀念館旁涉持刀傷人...疑巷內錯身起糾紛 男手臂撕裂傷送醫
9,國父紀念館旁腳踏車騎士亮刀傷人 險撞夫妻檔引爆口角



正在處理【運動】主題...
【運動】即時新聞結果 (共 20 筆)：


,運動新聞標題
0,中職／CNBLUE大巨蛋開唱鄭容和跑整圈 姜敏赫：悍將若缺投手找我
1,法網／絲薇托莉娜逆轉擊敗班西琪 生涯第6度闖法網8強
2,宜蘭縣運3項6人次破紀錄！游雅君、李慈雲表現亮眼 羅東奪總錦標冠軍
3,體操／唐嘉鴻單槓決賽發生失誤 世界挑戰盃排名第7
4,中職／林泓育生涯第4支再見安 打斷林詩翔跨季連24救援本土紀錄
5,參選中華籃協理事長鎩羽而歸 童子瑋：真心希望台灣籃球被更多人看見
6,法網／斯威雅蒂慘遭直落二擊敗 意外止步女單16強
7,中職／中信兄弟3戰敲34安 平野惠一盼年輕球員不怕犯錯
8,TPBL／衛冕只差一步 國王總教練驚爆「一定會請辭」
9,TPVL／天鷹直落3飛將 職排元年登頂



正在處理【科技】主題...
【科技】即時新聞結果 (共 20 筆)：


,科技新聞標題
0,博通推出業界首款端到端50G PON Edge AI產品組合 加速AI時代發展
1,iPhone 17玩《全明星街球派對》一鍵效能全開 主打輕鬆上手、爽快競技
2,不用再刪照片！iPhone三設定默默吃容量 網驚：按一鍵直接少11G
3,小米17T系列登台 徠卡5倍長焦、動態人像吸睛 信義區打造沉浸體驗館
4,他抱怨OLED螢幕撕膜後不耐刮⋯網揭「致命擦拭法」超刮螢幕
5,手機攜碼換eSIM的掃描時機為何？網推1作法實測好安心
6,Plaud Note Pro開箱！AI筆記工具超輕薄 實測遠處錄音、會議記錄、製圖生成驚豔
7,滿足各種需求 Xiaomi Watch S5 46mm、Xiaomi手環10 Pro NFC...
8,99％都做錯！iPhone這動作天天做 專家示警：電池更快耗損
9,中國出差順便撿便宜！網推薦3C必買清單「4K螢幕、機械鍵盤」原因曝光



正在處理【文教】主題...
【文教】即時新聞結果 (共 20 筆)：


,文教新聞標題
0,屏東國中小教甄試203名缺額創新高 今初試到考率7成多
1,處理校園親師生爭議 高雄推溝通先行、法律後援
2,台灣8部影片入圍慕尼黑兒童影視節 AI題材受關注
3,從夜間部到文化部長 李遠揭曾把父親氣到中風…返校演講提「文化森林」理念
4,中原大學量子金融嘉年華首登場 台日專家探應用新趨勢
5,不只內森！長榮大學永續發展國際學位學程招生13國學生 傳明年停招…各界惋惜
6,傳授「江湖行走」閱歷 吳良治世新大學畢典致詞：做對事情比做很多事情還重要
7,台大畢典最動人致詞！內森求學路惹淚無數人 最早就讀的大學學程申請停招
8,小學沒人、幼教超搶手！南投教甄兩樣情 國小師錄取率飆破4成6
9,李遠：政府應維護文化多樣性 培育森林非單一大樹
